In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


In [2]:
Cohort_Commo_Med = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Med_Cohort_S3")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# Rename column 'name' to 'full_name'
Cohort_Commo_Med = Cohort_Commo_Med.withColumnRenamed("mh_date", "MedicalHistory")

▸,:,


In [4]:
Cohort_Commo_Med.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- F25: long (nullable = true)
 |-- G12: long (n

In [11]:
# Get the list of column names
column_names = Cohort_Commo_Med.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

Number of columns: 2338


In [12]:
print(Cohort_Commo_Med.count())

152400


In [31]:
from pyspark.sql.functions import col, when, count, round

# Define age ranges
age_ranges = [
    (0, 5, '0-4'),
    (5, 10, '5-9'),
    (10, 15, '10-14'),
    (15, 20, '15-19'),
    (20, 25, '20-24'),
    (25, 30, '25-29'),
    (30, 35, '30-34'),
    (35, 40, '35-39'),
    (40, 45, '40-44'),
    (45, 50, '45-49'),
    (50, 55, '50-54'),
    (55, 60, '55-59'),
    (60, 65, '60-64'),
    (65, 70, '65-69'),
    (70, 75, '70-74'),
    (75, 80, '75-79'),
    (80, 85, '80-84'),
    (85, 90, '85-89'),
    (90, 95, '90-94'),
    (95, 100, '95-100'),
    (100, float('inf'), 'over_100')
]

# Convert age ranges to PySpark when expressions
age_conditions = [
    when((col("age_of_TBI_diagnosis") < upper) & (col("age_of_TBI_diagnosis") >= lower), range).alias("age_range_" + range.replace("-", "_"))
    for lower, upper, range in age_ranges
]

# Apply the conditions to create the age_range column
age_df = Cohort_Commo_Med.select(*age_conditions)

from pyspark.sql.functions import count

# Group by age_range and calculate counts
age_distribution = age_df.groupBy(*[col("age_range_" + range.replace("-", "_")) for _, _, range in age_ranges]) \
    .agg(count("*").alias("total_count_age"))

# Calculate percentage
total_records = Cohort_Commo_Med.count()
age_distribution = age_distribution.withColumn("percentage", 
                                               round((col("total_count_age") * 100.0) / total_records, 2))

# Show the results
age_distribution.orderBy(*[col("age_range_" + range.replace("-", "_")) for _, _, range in age_ranges]).show(truncate=False)


+-------------+-------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+----------------+------------------+---------------+----------+
|age_range_0_4|age_range_5_9|age_range_10_14|age_range_15_19|age_range_20_24|age_range_25_29|age_range_30_34|age_range_35_39|age_range_40_44|age_range_45_49|age_range_50_54|age_range_55_59|age_range_60_64|age_range_65_69|age_range_70_74|age_range_75_79|age_range_80_84|age_range_85_89|age_range_90_94|age_range_95_100|age_range_over_100|total_count_age|percentage|
+-------------+-------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+-

In [32]:
# Filter records with non-null age_range_0_4 column
age_distribution_non_null_0_4 = age_distribution.filter(age_distribution["age_range_0_4"].isNotNull())

# Show the filtered DataFrame
age_distribution_non_null_0_4.show(truncate=False)

+-------------+-------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+----------------+------------------+---------------+----------+
|age_range_0_4|age_range_5_9|age_range_10_14|age_range_15_19|age_range_20_24|age_range_25_29|age_range_30_34|age_range_35_39|age_range_40_44|age_range_45_49|age_range_50_54|age_range_55_59|age_range_60_64|age_range_65_69|age_range_70_74|age_range_75_79|age_range_80_84|age_range_85_89|age_range_90_94|age_range_95_100|age_range_over_100|total_count_age|percentage|
+-------------+-------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+-

In [7]:
# from pyspark.sql.functions import col, lit, concat_ws, when, round
# from pyspark.sql import DataFrame

# df = Cohort_Commo_Med

# # Define age ranges
# age_ranges = [
#     (0, 5, '0-4'),
#     (5, 10, '5-9'),
#     (10, 15, '10-14'),
#     (15, 20, '15-19'),
#     (20, 25, '20-24'),
#     (25, 30, '25-29'),
#     (30, 35, '30-34'),
#     (35, 40, '35-39'),
#     (40, 45, '40-44'),
#     (45, 50, '45-49'),
#     (50, 55, '50-54'),
#     (55, 60, '55-59'),
#     (60, 65, '60-64'),
#     (65, 70, '65-69'),
#     (70, 75, '70-74'),
#     (75, 80, '75-79'),
#     (80, 85, '80-84'),
#     (85, 90, '85-89'),
#     (90, 95, '90-94'),
#     (95, 100, '95-100'),
#     (100, float('inf'), 'over_100')
# ]

# # Update the stratification key creation logic to include age ranges
# age_conditions = [
#     when(
#         (col("age_of_TBI_diagnosis") >= lower) & (col("age_of_TBI_diagnosis") < upper),
#         range_label
#     ).alias(f"age_range_{range_label}")
#     for lower, upper, range_label in age_ranges
# ]

# # Apply the conditions to create the age_range column
# age_cols = [age_condition for age_condition in age_conditions]
# df = df.select(*age_cols, "gender", "race", "*")

# # Create a composite key for stratification including age range
# stratification_key_cols = [col(f"age_range_{range_label}") for _, _, range_label in age_ranges] + [col("gender"), col("race")]
# df = df.withColumn(
#     "stratification_key",
#     concat_ws("_", *stratification_key_cols)
# )

# # Define the fractions for each stratum (assuming 10% of each combination)
# fractions = df.groupBy("stratification_key").count().withColumn("fraction", lit(0.1))

# # Convert the fractions DataFrame to a dictionary suitable for sampleBy
# fraction_dict = {row["stratification_key"]: row["fraction"] for row in fractions.collect()}

# # Perform stratified sampling
# stratified_df = df.sampleBy("stratification_key", fractions=fraction_dict, seed=42)
# # Drop duplicate columns
# stratified_df = stratified_df.dropDuplicates()

# # Show the result
# stratified_df.show(5, truncate=False)

# # Count the number of records in the sampled DataFrame
# print(f"Number of records in stratified sample: {stratified_df.count()}")
from pyspark.sql.functions import col, lit, concat_ws, when, rand
# Assuming Cohort_Commo_Med DataFrame is already loaded
df = Cohort_Commo_Med

# Define age ranges
age_ranges = [
    (0, 5, '0-4'),
    (5, 10, '5-9'),
    (10, 15, '10-14'),
    (15, 20, '15-19'),
    (20, 25, '20-24'),
    (25, 30, '25-29'),
    (30, 35, '30-34'),
    (35, 40, '35-39'),
    (40, 45, '40-44'),
    (45, 50, '45-49'),
    (50, 55, '50-54'),
    (55, 60, '55-59'),
    (60, 65, '60-64'),
    (65, 70, '65-69'),
    (70, 75, '70-74'),
    (75, 80, '75-79'),
    (80, 85, '80-84'),
    (85, 90, '85-89'),
    (90, 95, '90-94'),
    (95, 100, '95-100'),
    (100, float('inf'), 'over_100')
]

# Update the stratification key creation logic to include age ranges
age_conditions = [
    when(
        (col("age_of_TBI_diagnosis") >= lower) & (col("age_of_TBI_diagnosis") < upper),
        range_label
    ).alias(f"age_range_{range_label}")
    for lower, upper, range_label in age_ranges
]

# Apply the conditions to create the age_range column
age_cols = [age_condition for age_condition in age_conditions]
df = df.withColumn("age_range", age_conditions[0])
for age_condition in age_conditions[1:]:
    df = df.withColumn("age_range", when(age_condition.isNotNull(), age_condition).otherwise(col("age_range")))

# Create a composite key for stratification including age range
stratification_key_cols = [col("age_range"), col("gender"), col("race")]
df = df.withColumn(
    "stratification_key",
    concat_ws("_", *stratification_key_cols)
)

# Define the fractions for each stratum (assuming 10% of each combination)
fractions_df = df.groupBy("stratification_key").count().withColumn("fraction", lit(0.1))

# Join the original DataFrame with the fractions DataFrame
df = df.join(fractions_df, on="stratification_key")

# Perform stratified sampling using the fraction column
spark.conf.set("spark.sql.shuffle.partitions", "10")
df = df.withColumn("rand", rand(seed=42))
stratified_df = df.filter(col("rand") < col("fraction"))

# Drop duplicate columns (if any)
stratified_df = stratified_df.drop("count", "fraction", "rand")

# Show the result
stratified_df.show(5, truncate=False)

# Count the number of records in the sampled DataFrame
print(f"Number of records in stratified sample: {stratified_df.count()}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of records in stratified sample: 15258


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
from pyspark.sql.functions import col, lit, concat_ws
# Assuming Cohort_Commo_Med DataFrame is already loaded
df = Cohort_Commo_Med

# Display the schema and some sample data
df.printSchema()
df.show(5, truncate=False)

# Create a composite key for stratification
df = df.withColumn("stratification_key", concat_ws("_", col("age_of_TBI_diagnosis"), col("gender"), col("race")))

# Define the fractions for each stratum
# For simplicity, let's assume we want 10% of each combination
fractions = df.groupBy("stratification_key").count().withColumn("fraction", lit(0.1))

# Convert the fractions DataFrame to a dictionary suitable for sampleBy
fraction_dict = { row["stratification_key"]: row["fraction"] for row in fractions.collect() }

# Perform stratified sampling
stratified_df = df.sampleBy("stratification_key", fractions=fraction_dict, seed=42)
# Drop duplicate columns
stratified_df = stratified_df.dropDuplicates()
# Show the result
stratified_df.show(5, truncate=False)

# Count the number of records in the sampled DataFrame
print(f"Number of records in stratified sample: {stratified_df.count()}")

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- F25: long (nullable = true)
 |-- G12: long (n

+------------------------------------+----------+-------------------------+-------------------------+--------------------+--------------------+----------+------+--------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+

+------------------------------------+----------+-------------------------+-------------------------+--------------------+--------------------+----------+------+--------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+

Number of records in stratified sample: 15817


In [10]:
from pyspark.sql.functions import col, lit, concat_ws, count, round

# Calculate the total count of records in the stratified DataFrame
total_count = stratified_df.count()

# Function to calculate count and percentage for a given column
def calculate_distribution(df, column_name):
    distribution = (
        df.groupBy(column_name)
          .agg(
              count("*").alias("count"),
              (count("*") / total_count * 100).alias("percentage")
          )
          .orderBy(column_name)
    )
    return distribution

# Calculate and show distribution for age_of_TBI_diagnosis
age_of_TBI_diagnosis_distribution = calculate_distribution(stratified_df, "age_of_TBI_diagnosis")
age_of_TBI_diagnosis_distribution = age_of_TBI_diagnosis_distribution.withColumn("percentage", round(col("percentage"), 2))
print("Distribution for age_of_TBI_diagnosis:")
age_of_TBI_diagnosis_distribution.show(truncate=False)

# Calculate and show distribution for race
race_distribution = calculate_distribution(stratified_df, "race")
race_distribution = race_distribution.withColumn("percentage", round(col("percentage"), 2))
print("Distribution for race:")
race_distribution.show(truncate=False)

# Calculate and show distribution for gender
gender_distribution = calculate_distribution(stratified_df, "gender")
gender_distribution = gender_distribution.withColumn("percentage", round(col("percentage"), 2))
print("Distribution for gender:")
gender_distribution.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------+-----+----------+
|gender      |count|percentage|
+------------+-----+----------+
|Female      |7244 |47.48     |
|Male        |8026 |52.6      |
|other_gender|8    |0.05      |
+------------+-----+----------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
from pyspark.sql.functions import col, lit, concat_ws, count, round

# Count the number of records in the sampled DataFrame
total_count = stratified_df.count()

# Function to calculate count and percentage for a given column
def calculate_distribution(df, column_name):
    distribution = (
        df.groupBy(column_name)
          .agg(
              count("*").alias("count"),
              (count("*") / total_count * 100).alias("percentage")
          )
          .orderBy(column_name)
    )
    return distribution

# Calculate distribution statistics for all categories
distribution_stats = (
    stratified_df.groupBy("age_of_TBI_diagnosis", "race", "gender")
                 .agg(
                     count("*").alias("count"),
                     (count("*") / total_count * 100).alias("percentage")
                 )
                 .orderBy("age_of_TBI_diagnosis", "race", "gender")
)

# Round the percentage column
distribution_stats = distribution_stats.withColumn("percentage", round(col("percentage"), 2))

# Show the distribution statistics
print("Distribution statistics for all categories:")
distribution_stats.show(truncate=False)

Distribution statistics for all categories:
+--------------------+---------------+------+-----+----------+
|age_of_TBI_diagnosis|race           |gender|count|percentage|
+--------------------+---------------+------+-----+----------+
|0.0058243725        |Other_race     |Male  |1    |0.01      |
|0.006160394166666666|Other_race     |Female|1    |0.01      |
|0.006160394166666666|Other_race     |Male  |1    |0.01      |
|0.007280465833333334|White          |Female|1    |0.01      |
|0.013209845833333332|White          |Female|1    |0.01      |
|0.015681003333333336|White          |Male  |1    |0.01      |
|0.023577509166666667|White          |Female|1    |0.01      |
|0.030129928333333333|Other_race     |Female|1    |0.01      |
|0.03841845916666667 |White          |Female|1    |0.01      |
|0.04883512583333333 |White          |Male  |1    |0.01      |
|0.05723566333333333 |Other_race     |Male  |1    |0.01      |
|0.065076165         |White          |Male  |1    |0.01      |
|0.07431675

In [45]:
print(distribution_stats.count())

15709


In [46]:
print(stratified_df.count())

15817


In [22]:
print(stratified_df.select("personid").distinct().count())

15817


In [57]:
# Get the list of column names
column_names = stratified_df.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

Number of columns: 2339


In [8]:
stratified_df.printSchema()

▸,:,


root
 |-- stratification_key: string (nullable = false)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)

In [9]:
from pyspark.sql.functions import col

# Define a function to check for duplicate columns
def has_duplicate_columns(df):
    # Get distinct columns and all columns
    distinct_cols = set([col_name.lower() for col_name in df.columns])
    all_cols = [col_name.lower() for col_name in df.columns]
    
    # If the length of distinct columns is less than all columns, there are duplicates
    return len(distinct_cols) < len(all_cols)

# Check if stratified_df has duplicate columns
if has_duplicate_columns(stratified_df):
    print("stratified_df has duplicate columns.")
else:
    print("stratified_df does not have duplicate columns.")

▸,:,


stratified_df does not have duplicate columns.


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, lit, row_number
from pyspark.sql.window import Window
stratification_column = "race"
balancing_column = "gender"
total_population = Cohort_Commo_Med.count()
sample_fraction = 0.1  # Define the fraction of the data to be sampled

strata_proportions = (Cohort_Commo_Med
                      .groupBy(stratification_column)
                      .count()
                      .withColumn("proportion", col("count") / total_population))

strata_proportions.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+---------------+------+--------------------+
|race           |count |proportion          |
+---------------+------+--------------------+
|Native American|2047  |0.013431758530183727|
|Other_race     |31664 |0.20776902887139106 |
|White          |109105|0.7159120734908136  |
|Hispanic       |5626  |0.03691601049868767 |
|Black          |2009  |0.013182414698162729|
|Asian          |1949  |0.012788713910761154|
+---------------+------+--------------------+



<IPython.core.display.Javascript object>

In [48]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, lit, row_number
from pyspark.sql.window import Window
stratification_column = "race"
balancing_column = "gender"
total_population = stratified_df.count()
sample_fraction = 0.1  # Define the fraction of the data to be sampled

strata_proportions = (Cohort_Commo_Med
                      .groupBy(stratification_column)
                      .count()
                      .withColumn("proportion", col("count") / total_population))

strata_proportions.show(truncate=False)

+---------------+------+-------------------+
|race           |count |proportion         |
+---------------+------+-------------------+
|Native American|2047  |0.12941771511664665|
|Other_race     |31664 |2.001896693431118  |
|White          |109105|6.897957893405829  |
|Hispanic       |5626  |0.35569324144907377|
|Black          |2009  |0.1270152367705633 |
|Asian          |1949  |0.12322184990832648|
+---------------+------+-------------------+



In [8]:
stratified_df.printSchema()

▸,:,


root
 |-- age_range_0-4: string (nullable = true)
 |-- age_range_5-9: string (nullable = true)
 |-- age_range_10-14: string (nullable = true)
 |-- age_range_15-19: string (nullable = true)
 |-- age_range_20-24: string (nullable = true)
 |-- age_range_25-29: string (nullable = true)
 |-- age_range_30-34: string (nullable = true)
 |-- age_range_35-39: string (nullable = true)
 |-- age_range_40-44: string (nullable = true)
 |-- age_range_45-49: string (nullable = true)
 |-- age_range_50-54: string (nullable = true)
 |-- age_range_55-59: string (nullable = true)
 |-- age_range_60-64: string (nullable = true)
 |-- age_range_65-69: string (nullable = true)
 |-- age_range_70-74: string (nullable = true)
 |-- age_range_75-79: string (nullable = true)
 |-- age_range_80-84: string (nullable = true)
 |-- age_range_85-89: string (nullable = true)
 |-- age_range_90-94: string (nullable = true)
 |-- age_range_95-100: string (nullable = true)
 |-- age_range_over_100: string (nullable = true)
 |-- gen

In [11]:
stratified_df.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_StrataSampledraft')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
print(stratified_df.select("personid").distinct().count())

15817


In [60]:
# Get the list of column names
column_names = stratified_df.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

Number of columns: 2339


In [61]:
stratified_df.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- F25: long (nullable = true)
 |-- G12: long (n

In [2]:
Cohort_Draft = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_StrataSampledraft")

In [3]:
print(Cohort_Draft.select("personid").distinct().count())

15278


In [21]:
Cohort_Draft.printSchema()

▸,:,


root
 |-- stratification_key: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)


In [3]:
Epilepsy_Cohort_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Cohort_toStack")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# Perform the join on personid
joined_df = Cohort_Draft.join(Epilepsy_Cohort_Lab, on="personid", how="inner")

# Select the required columns: personid from Cohort_Draft and all columns except personid from Epilepsy_Cohort_Lab
# Get the list of columns from Epilepsy_Control_Lab except personid
columns_from_lab = [col for col in Epilepsy_Cohort_Lab.columns if col != "personid"]

# Select personid from Cohort_Draft and the remaining columns from Epilepsy_Cohort_Lab
result_df = joined_df.select("personid", *columns_from_lab)

# Show the result DataFrame
result_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------+--------------------------+-----------+
|personid                            |labcode|New_updated_Interpretation|servicedate|
+------------------------------------+-------+--------------------------+-----------+
|52ad17e0-58fc-4b53-ade5-1684cd0808d9|13303-3|Normal                    |2022-04-08 |
|49eb4a3d-db55-4a24-a6f5-753f9a40d262|15283-5|null                      |2016-12-08 |
|6b5131c2-31ff-4b1c-af34-530c776f15a6|15283-5|null                      |2016-07-28 |
|222b7933-571f-4540-a324-89857459bc2e|15283-5|null                      |2016-12-31 |
|08d89c3e-3ddd-409a-9502-5010aec6c9f5|15283-5|null                      |2014-10-21 |
|b8c2f526-89b7-4513-b1f8-ae7065f8f0ce|15283-5|null                      |2017-09-04 |
|81c86e57-d8e3-424b-b7bd-26867ff32331|18478-8|null                      |2017-01-17 |
|44c4929f-90c6-4c87-a4b4-2badb85859ce|19319-3|Normal                    |2019-05-12 |
|088e6267-3f75-4f79-9c45-1c19effd8017|19319-3|Abnormal

<IPython.core.display.Javascript object>

In [6]:
from pyspark.sql import functions as F
# reparNum = 0  # Assuming reparNum is defined somewhere in your code
reparNum = result_df.rdd.getNumPartitions()
Epilepsy_Cohort_Lab_repart = result_df.repartition(reparNum)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:

# Pivot the labcode column values into individual columns and repartition
pivoted_lab_df1 = (
    Epilepsy_Cohort_Lab_repart.groupby('personid')
    .pivot('labcode')
    .agg(F.first('New_updated_Interpretation'))
)
# pivoted_lab_df = pivoted_lab_df.withColumnRenamed('personid', 'pivoted_personid')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
pivoted_lab_df1.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_StratifiedSampleLab1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
pivoted_lab_df1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_StratifiedSampleLab1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
from pyspark.sql import functions as F
# reparNum = 0  # Assuming reparNum is defined somewhere in your code
reparNum = pivoted_lab_df1.rdd.getNumPartitions()
pivoted_lab_df1_repart = pivoted_lab_df1.repartition(reparNum)

▸,:,


In [6]:
from pyspark.sql.functions import col, count, when, lit

# Function to calculate the percentage of nulls in each column
def calculate_null_percentage(df):
    total_rows = df.count()
    null_counts = df.select([count(when(col(c).isNull(), 1)).alias(c) for c in df.columns])
    null_percentage = null_counts.select([(col(c) / total_rows).alias(c) for c in null_counts.columns])
    return null_percentage

# Calculate null percentages for each column
null_percentages_df = calculate_null_percentage(pivoted_lab_df1_repart)

# Create a DataFrame to hold the threshold value
threshold_df = spark.createDataFrame([(0.9,)], ["threshold"])

# Create a condition to check which columns have null percentage greater than 90%
conditions = [when(col(c) > threshold_df.threshold, lit(c)).alias(c) for c in null_percentages_df.columns]

# Apply the conditions and filter out nulls to get the columns to drop
columns_to_drop_df = null_percentages_df.crossJoin(threshold_df).select(conditions)

# Collect the column names to drop from the DataFrame
columns_to_drop = [row[c] for row in columns_to_drop_df.collect() for c in columns_to_drop_df.columns if row[c] is not None]

# Drop those columns from the DataFrame
filtered_Cohort_df = pivoted_lab_df1_repart.drop(*columns_to_drop)

# Display the result
print(f"Total number of columns with more than 90% nulls: {len(columns_to_drop)}")
print(f"Columns dropped: {columns_to_drop}")

# Show the new DataFrame
filtered_Cohort_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------+-------+-------+---------+-------+--------+-------+------+------+------+------+-------+-------+-------+------+------+------+------+--------+------+------+------+-------+------+------+------+------+--------+------+------+------+------+------+-------+------+------+------+-------+-------+------+------+-------+-------+-------+------+------+--------+------+--------+--------+--------+--------+------+------+------+--------+-------+------+------+------+------+------+------+------+------+------+-------+------+------+------+------+------+------+------+------+------+------+------+------+--------+------+-------+
|personid                            |10466-1|10839-9|11579-0|119297000|13457-7|13945-1 |14979-9|1742-6|1743-4|1751-7|1759-0|17861-6|19123-9|19161-9|1920-8|1968-7|1975-2|2028-9|20454-5 |2075-0|2085-9|2093-3|21000-5|2157-6|2160-0|2339-0|2345-7|2514-8  |2571-8|2777-1|2823-3|2885-2|2951-2|30239-8|3040-3|3094-0|3097-3|32623-1|33037-3|4544-3|4548-4|

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# Get the list of column names
column_names = filtered_Cohort_df.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 83


In [39]:
# Get the list of column names
column_names = pivoted_lab_df1.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 3715


In [11]:
from pyspark.sql.functions import col

# Perform left join
joined_df = Cohort_Draft.join(filtered_Cohort_df, 
                                    Cohort_Draft.personid == filtered_Cohort_df.personid,
                                    "left")

# Selecting columns from both DataFrames
selected_columns = [Cohort_Draft[col] for col in Cohort_Draft.columns] + \
                   [filtered_Cohort_df[col] for col in filtered_Cohort_df.columns if col != "personid"]

# Selecting columns from the joined DataFrame
result_df2 = joined_df.select(selected_columns)

▸,:,


In [12]:
# Get the list of column names
column_names = result_df2.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 2422


In [13]:
print(result_df2.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

15278


<IPython.core.display.Javascript object>

In [14]:
result_df2.printSchema()

▸,:,


root
 |-- stratification_key: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)


In [16]:
result_df2.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_StratifiedSample2')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
Epilepsy_Cohort_SS = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_StratifiedSample2")

In [17]:
from pyspark.sql.functions import col, count, when

# Calculate the total number of rows
total_rows = pivoted_lab_df1.count()

# Function to calculate the percentage of nulls in each column
def calculate_null_percentage(df, total_rows):
    null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
    null_percentage_row = null_counts.select([(col(c) / total_rows).alias(c) for c in null_counts.columns]).collect()[0]
    return {col_name: null_percentage_row[col_name] for col_name in df.columns}

# Calculate null percentages for each column
null_percentages = calculate_null_percentage(pivoted_lab_df1, total_rows)

# Identify columns with more than 90% nulls
columns_to_drop = [c for c in pivoted_lab_df1.columns if null_percentages[c] > 0.9]

# Drop those columns from the DataFrame
filtered_CohortLab_df = pivoted_lab_df1.drop(*columns_to_drop)

# Display the result
print(f"Total number of columns with more than 90% nulls: {len(columns_to_drop)}")
# print(f"Columns dropped: {columns_to_drop}")

# Show the new DataFrame
filtered_CohortLab_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------+--------+-------+---------+-------+-------+-------+------+------+------+------+-------+-------+--------+------+------+------+------+-------+------+------+------+-------+------+------+------+--------+------+------+------+------+------+------+-------+------+------+------+-------+-------+------+------+-------+-------+-------+--------+--------+------+------+--------+--------+--------+------+------+--------+------+------+-------+------+------+------+------+------+------+------+------+------+-------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+
|personid                            |10466-1|10839-9 |11579-0|119297000|13457-7|13945-1|14979-9|1742-6|1743-4|1751-7|1759-0|17861-6|19123-9|19161-9 |1920-8|1968-7|1975-2|2028-9|20454-5|2075-0|2085-9|2093-3|21000-5|2157-6|2160-0|2339-0|2345-7  |2514-8|2571-8|2777-1|2823-3|2885-2|2951-2|30239-8|3040-3|3094-0|3097-3|32623-1|33037-3|4544-3|4548-4|48

<IPython.core.display.Javascript object>

In [18]:
from pyspark.sql.functions import col

# Perform left join
joined_df = Cohort_Draft.join(filtered_CohortLab_df, 
                                    Cohort_Draft.personid == filtered_CohortLab_df.personid,
                                    "left")

# Selecting columns from both DataFrames
selected_columns = [Cohort_Draft[col] for col in Cohort_Draft.columns] + \
                   [filtered_CohortLab_df[col] for col in filtered_CohortLab_df.columns if col != "personid"]

# Selecting columns from the joined DataFrame
result_df2 = joined_df.select(selected_columns)

▸,:,


In [8]:
# Get the list of column names
column_names = result_df2.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 2422


In [9]:
result_df2.printSchema()

▸,:,


root
 |-- stratification_key: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)


In [19]:
result_df2.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_StratifiedSample')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
Epilepsy_Cohort_SS = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Cohort_StratifiedSample1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
# Get the list of column names
column_names = Epilepsy_Cohort_SS.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 6054


In [24]:
Epilepsy_Cohort_SS.printSchema()

▸,:,


root
 |-- stratification_key: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- EPI_date: string (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- age_at_EPI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- Z21: long (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- L65: long (nullable = true)


In [17]:
Epilepsy_Control_Commo_Med = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Demo_Como_Med_Control_S3")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
Epilepsy_Control_Lab = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Cleaned_Lab_Control_toStack")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
from pyspark.sql.functions import col, lit, concat_ws, when, rand
# Assuming Cohort_Commo_Med DataFrame is already loaded
df = Epilepsy_Control_Commo_Med

# Define age ranges
age_ranges = [
    (0, 5, '0-4'),
    (5, 10, '5-9'),
    (10, 15, '10-14'),
    (15, 20, '15-19'),
    (20, 25, '20-24'),
    (25, 30, '25-29'),
    (30, 35, '30-34'),
    (35, 40, '35-39'),
    (40, 45, '40-44'),
    (45, 50, '45-49'),
    (50, 55, '50-54'),
    (55, 60, '55-59'),
    (60, 65, '60-64'),
    (65, 70, '65-69'),
    (70, 75, '70-74'),
    (75, 80, '75-79'),
    (80, 85, '80-84'),
    (85, 90, '85-89'),
    (90, 95, '90-94'),
    (95, 100, '95-100'),
    (100, float('inf'), 'over_100')
]

# Update the stratification key creation logic to include age ranges
age_conditions = [
    when(
        (col("age_of_TBI_diagnosis") >= lower) & (col("age_of_TBI_diagnosis") < upper),
        range_label
    ).alias(f"age_range_{range_label}")
    for lower, upper, range_label in age_ranges
]

# Apply the conditions to create the age_range column
age_cols = [age_condition for age_condition in age_conditions]
df = df.withColumn("age_range", age_conditions[0])
for age_condition in age_conditions[1:]:
    df = df.withColumn("age_range", when(age_condition.isNotNull(), age_condition).otherwise(col("age_range")))

# Create a composite key for stratification including age range
stratification_key_cols = [col("age_range"), col("gender"), col("race")]
df = df.withColumn(
    "stratification_key",
    concat_ws("_", *stratification_key_cols)
)

# Define the fractions for each stratum (assuming 10% of each combination)
fractions_df = df.groupBy("stratification_key").count().withColumn("fraction", lit(0.1))

# Join the original DataFrame with the fractions DataFrame
df = df.join(fractions_df, on="stratification_key")

# Perform stratified sampling using the fraction column
spark.conf.set("spark.sql.shuffle.partitions", "10")
df = df.withColumn("rand", rand(seed=42))
stratified_df = df.filter(col("rand") < col("fraction"))

# Drop duplicate columns (if any)
stratified_df = stratified_df.drop("count", "fraction", "rand")

# Show the result
stratified_df.show(5, truncate=False)

# Count the number of records in the sampled DataFrame
print(f"Number of records in stratified sample: {stratified_df.count()}")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of records in stratified sample: 100950


<IPython.core.display.Javascript object>

In [4]:
from pyspark.sql.functions import col, lit, concat_ws, count, round

# Calculate the total count of records in the stratified DataFrame
total_count = stratified_df.count()

# Function to calculate count and percentage for a given column
def calculate_distribution(df, column_name):
    distribution = (
        df.groupBy(column_name)
          .agg(
              count("*").alias("count"),
              (count("*") / total_count * 100).alias("percentage")
          )
          .orderBy(column_name)
    )
    return distribution

# Calculate and show distribution for age_of_TBI_diagnosis
age_of_TBI_diagnosis_distribution = calculate_distribution(stratified_df, "age_of_TBI_diagnosis")
age_of_TBI_diagnosis_distribution = age_of_TBI_diagnosis_distribution.withColumn("percentage", round(col("percentage"), 2))
print("Distribution for age_of_TBI_diagnosis:")
age_of_TBI_diagnosis_distribution.show(truncate=False)

# Calculate and show distribution for race
race_distribution = calculate_distribution(stratified_df, "race")
race_distribution = race_distribution.withColumn("percentage", round(col("percentage"), 2))
print("Distribution for race:")
race_distribution.show(truncate=False)

# Calculate and show distribution for gender
gender_distribution = calculate_distribution(stratified_df, "gender")
gender_distribution = gender_distribution.withColumn("percentage", round(col("percentage"), 2))
print("Distribution for gender:")
gender_distribution.show(truncate=False)

Distribution for age_of_TBI_diagnosis:
+---------------------+-----+----------+
|age_of_TBI_diagnosis |count|percentage|
+---------------------+-----+----------+
|0.0                  |4    |0.0       |
|0.0031362008333333333|1    |0.0       |
|0.0032482074999999997|2    |0.0       |
|0.0033602149999999997|1    |0.0       |
|0.0034722225000000002|1    |0.0       |
|0.0036551675         |1    |0.0       |
|0.0044914874999999995|1    |0.0       |
|0.004592294166666667 |2    |0.0       |
|0.004816308333333333 |1    |0.0       |
|0.0058243725         |1    |0.0       |
|0.005936380000000001 |1    |0.0       |
|0.007168459166666666 |1    |0.0       |
|0.007174059166666667 |1    |0.0       |
|0.00750448           |1    |0.0       |
|0.007650089999999999 |1    |0.0       |
|0.008624551666666666 |1    |0.0       |
|0.009192055          |1    |0.0       |
|0.009856630833333333 |1    |0.0       |
|0.010056376666666667 |1    |0.0       |
|0.010179585          |1    |0.0       |
+-----------------

In [5]:
print(stratified_df.select("personid").distinct().count())

101723


In [5]:
# Get the list of column names
column_names = stratified_df.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

Number of columns: 2385


In [8]:
# Drop duplicate columns
stratified_df = stratified_df.dropDuplicates()

In [28]:
stratified_df.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_StrataSampledraft')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
Control_Draft = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_StrataSampledraft")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
# Rename column 'name' to 'full_name'
Control_Draft = Control_Draft.withColumnRenamed("mh_date", "MedicalHistory")

▸,:,


In [24]:
Control_Draft.printSchema()

▸,:,


root
 |-- stratification_key: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- latest_diagdate: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Z21: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- X04: long (nullable = true)
 |-- F25: lo

In [28]:
# List of columns to be excluded
exclude_columns = ['stratification_key', 'personid', 'birthdate', 'EPI_date', 'TBI_date', 'age_of_TBI_diagnosis', 'age_at_EPI_diagnosis', 'race', 'gender', 'MedicalHistory', 'age_range']

# Collect the column names except the ones in exclude_columns
column_names1 = [col for col in Cohort_Draft.columns if col not in exclude_columns]

# Print the result
print(column_names1)

▸,:,


['Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B99', 'Q32', 'X82', 'O89', 'Q73', 'R01', 'R30', 'X50', 'T07', 'E66', 'L97', 'O34', 'I61', 'M93', 'D3A', 'P59', 'L90', 'Q33', 'P74', 'Y69', 'Q16', 'Z98', 'O42', 'I49', 'T63', 'I96', 'D29', 'B30', 'N02', 'I22', 'Q87', 'F04', 'C03', 'H60', 'N90', 'B85', 'B50', 'E84', 'G64', 'B06', 'A35', 'J30', 'K80', 'J69'

In [29]:
# List of columns to be excluded
exclude_columns = ['stratification_key', 'personid', 'birthdate', 'latest_diagdate', 'TBI_date', 'age_of_TBI_diagnosis', 'race', 'gender', 'MedicalHistory', 'age_range']

# Collect the column names except the ones in exclude_columns
column_names2 = [col for col in Control_Draft.columns if col not in exclude_columns]

# Print the result
print(column_names2)

▸,:,


['M19', 'S68', 'Z21', 'Y30', 'A23', 'B05', 'H82', 'V89', 'R16', 'I31', 'Q61', 'V72', 'O12', 'X76', 'S39', 'Z12', 'L65', 'X04', 'F25', 'E02', 'G12', 'B79', 'M54', 'S60', 'F32', 'B34', 'E44', 'T36', 'E83', 'Q65', 'R71', 'Z19', 'Z64', 'D66', 'F21', 'D81', 'Q14', 'J60', 'P00', 'B01', 'C22', 'Z49', 'V24', 'R13', 'L74', 'E56', 'I44', 'O68', 'H59', 'D28', 'I01', 'R06', 'K56', 'O14', 'C78', 'R37', 'A46', 'Y02', 'P72', 'S00', 'K62', 'R80', 'K40', 'M92', 'K03', 'M46', 'P13', 'C24', 'J63', 'C95', 'X13', 'CP4', 'D16', 'I21', 'H51', 'Z08', 'O11', 'C77', 'I06', 'P96', 'Q62', 'J93', 'F01', 'F99', 'H34', 'A92', 'E35', 'W42', 'Z91', 'I63', 'G96', 'B39', 'W58', 'Q80', 'N99', 'R47', 'I77', 'D21', 'L27', 'B99', 'J81', 'O89', 'Q32', 'X82', 'Q73', 'R30', 'P59', 'R01', 'T07', 'E66', 'X50', 'L97', 'I61', 'O34', 'L90', 'Q33', 'P74', 'M93', 'Q16', 'D3A', 'Y69', 'T63', 'Z98', 'I49', 'B30', 'O42', 'N02', 'I96', 'D29', 'I22', 'Q87', 'F04', 'C03', 'H60', 'N90', 'B85', 'B50', 'E84', 'G64', 'A35', 'B06', 'J30', 'J69'

In [30]:
# # Convert lists to sets and find the intersection
# Final_Columns = list(set(column_names1) & set(column_names2))

# # Print the result
# print(Final_Columns)
# Extract matched items
Final_Columns = [col for col in column_names1 if col in column_names2]

# Print the result
print(Final_Columns)

▸,:,


['Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B99', 'Q32', 'X82', 'O89', 'Q73', 'R01', 'R30', 'X50', 'T07', 'E66', 'L97', 'O34', 'I61', 'M93', 'D3A', 'P59', 'L90', 'Q33', 'P74', 'Y69', 'Q16', 'Z98', 'O42', 'I49', 'T63', 'I96', 'D29', 'B30', 'N02', 'I22', 'Q87', 'F04', 'C03', 'H60', 'N90', 'B85', 'B50', 'E84', 'G64', 'B06', 'A35', 'J30', 'K80', 'J69'

In [31]:
# Extract matched items
Final_Columns = [col for col in column_names1 if col in column_names2]

# List of items to add at the beginning
additional_columns = ['stratification_key', 'personid', 'birthdate', 'latest_diagdate', 'TBI_date', 'age_of_TBI_diagnosis', 'race', 'gender', 'MedicalHistory', 'age_range']

# Add the additional columns to the beginning of Final_Columns
Final_Columns = additional_columns + Final_Columns

# Print the result
print(Final_Columns)

▸,:,


['stratification_key', 'personid', 'birthdate', 'latest_diagdate', 'TBI_date', 'age_of_TBI_diagnosis', 'race', 'gender', 'MedicalHistory', 'age_range', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B99', 'Q32', 'X82', 'O89', 'Q73', 'R01', 'R30', 'X50', 'T07', 'E66', 'L97', 'O34', 'I61', 'M93', 'D3A', 'P59', 'L90', 'Q33', 'P74', 'Y69', 'Q16', 'Z98', '

In [32]:
# Filter the columns
filtered_columns = [col for col in Control_Draft.columns if col in Final_Columns]

# Select only the filtered columns from Control_Draft
final_df = Control_Draft.select(filtered_columns)

# Show the result
final_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-----------------------+------------------------------------+----------+-------------------------+-------------------------+--------------------+----------+------+--------------+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+-

<IPython.core.display.Javascript object>

In [30]:
print(final_df.count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

100794


<IPython.core.display.Javascript object>

In [31]:
final_df.printSchema()

▸,:,


root
 |-- stratification_key: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- latest_diagdate: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Z21: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- X04: long (nullable = true)
 |-- F25: lo

In [33]:
# Get the list of column names
column_names = final_df.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 2313


In [34]:
from pyspark.sql.functions import col

# Filter the DataFrame
filtered_df = result_df.filter(result_df.labcode.isin(column_names))

# Show the filtered result
filtered_df.show(truncate=False)

▸,:,


NameError: name 'result_df' is not defined

In [35]:
# Perform the join on personid
joined_df = Control_Draft.join(Epilepsy_Control_Lab, on="personid", how="inner")

# Select the required columns: personid from Cohort_Draft and all columns except personid from Epilepsy_Cohort_Lab
# Get the list of columns from Epilepsy_Control_Lab except personid
columns_from_lab = [col for col in Epilepsy_Control_Lab.columns if col != "personid"]

# Select personid from Cohort_Draft and the remaining columns from Epilepsy_Cohort_Lab
result_df = joined_df.select("personid", *columns_from_lab)

# Show the result DataFrame
result_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------+--------------------------+-----------+
|personid                            |labcode|New_updated_Interpretation|servicedate|
+------------------------------------+-------+--------------------------+-----------+
|5d4ab027-e85e-4a58-b7c8-7eec4b42ad78|15283-5|null                      |2019-07-25 |
|229956ad-77a9-4fcf-bc5c-cde70357f795|15283-5|null                      |2014-11-26 |
|c6a75e40-0451-4328-903b-e2571b531b94|15283-5|Low                       |2019-05-08 |
|0d0e7c64-27b2-4472-a8c7-91e05c65f9f7|15283-5|High                      |2019-12-23 |
|4fb8ea3e-fa29-422d-b93b-51e5652ad15d|15283-5|null                      |2013-05-21 |
|69008533-76fc-411e-b0d0-5162ee09f394|15283-5|Normal                    |2021-05-06 |
|1830f9ab-ecae-452f-a32f-d408d503183f|15283-5|null                      |2017-12-20 |
|ba95888d-1c9a-4004-bd3f-3947469898f9|15283-5|null                      |2021-09-14 |
|6facc9ff-a0aa-4486-8239-9a556de07037|15283-5|null    

<IPython.core.display.Javascript object>

In [36]:
print(result_df.select("labcode").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

4432


<IPython.core.display.Javascript object>

In [37]:
column_names = [col for col in pivoted_lab_df1.columns if col != 'personid']

# Print the result
print(column_names)

▸,:,


['1005-8', '1006-6', '1007-4', '10328-3', '10329-1', '10333-3', '10334-1', '10335-8', '10338-2', '10346-5', '10353-1', '10362-2', '10368-9', '10374-7', '10376-2', '10377-0', '10378-8', '10379-6', '10380-4', '10381-2', '10449-7', '10451-3', '10466-1', '1048003', '10501-5', '10524-7', '10535-3', '10552-8', '10568-4', '10573-4', '10580-9', '10622-9', '10624-5', '10676-5', '10834-0', '10835-7', '10839-9', '10886-0', '10895-1', '10900-9', '10907-4', '10912-4', '10976-9', '10998-3', '11004-9', '11006-4', '11011-4', '11013-0', '11024-7', '11034-6', '11038-7', '11043-7', '11046-0', '11050-2', '11051-0', '11054-4', '11060-1', '11067-6', '11071-8', '11083-3', '11090-8', '11103-9', '11106-2', '11111-2', '11112-0', '11114-6', '11118-7', '11125-2', '11127-8', '11134-4', '11145-0', '11154-2', '11156-7', '11183-1', '11211-0', '11218-5', '11235-9', '11246-6', '11253-2', '11256-5', '11258-1', '11259-9', '11266-4', '11271-4', '11272-2', '11273-0', '11274-8', '11277-1', '11279-7', '11483-5', '11546-9', '

In [38]:
from pyspark.sql.functions import col

# Filter the DataFrame
filtered_df = result_df.filter(result_df.labcode.isin(column_names))

# Show the filtered result
filtered_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------+--------------------------+-----------+
|personid                            |labcode|New_updated_Interpretation|servicedate|
+------------------------------------+-------+--------------------------+-----------+
|5d4ab027-e85e-4a58-b7c8-7eec4b42ad78|15283-5|null                      |2019-07-25 |
|229956ad-77a9-4fcf-bc5c-cde70357f795|15283-5|null                      |2014-11-26 |
|c6a75e40-0451-4328-903b-e2571b531b94|15283-5|Low                       |2019-05-08 |
|0d0e7c64-27b2-4472-a8c7-91e05c65f9f7|15283-5|High                      |2019-12-23 |
|4fb8ea3e-fa29-422d-b93b-51e5652ad15d|15283-5|null                      |2013-05-21 |
|69008533-76fc-411e-b0d0-5162ee09f394|15283-5|Normal                    |2021-05-06 |
|1830f9ab-ecae-452f-a32f-d408d503183f|15283-5|null                      |2017-12-20 |
|ba95888d-1c9a-4004-bd3f-3947469898f9|15283-5|null                      |2021-09-14 |
|6facc9ff-a0aa-4486-8239-9a556de07037|15283-5|null    

<IPython.core.display.Javascript object>

In [39]:
print(filtered_df.select("labcode").distinct().count())

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

3242


<IPython.core.display.Javascript object>

In [15]:
from pyspark.sql import functions as F
# reparNum = 0  # Assuming reparNum is defined somewhere in your code
reparNum = filtered_df.rdd.getNumPartitions()
Epilepsy_Control_Lab_repart = filtered_df.repartition(reparNum)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:

# Pivot the labcode column values into individual columns and repartition
pivoted_lab_df = (
    Epilepsy_Control_Lab_repart.groupby('personid')
    .pivot('labcode')
    .agg(F.first('New_updated_Interpretation'))
)
# pivoted_lab_df = pivoted_lab_df.withColumnRenamed('personid', 'pivoted_personid')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
pivoted_lab_df.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_StratifiedSampleLab1')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
pivoted_lab_df = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_StratifiedSampleLab1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
from pyspark.sql import functions as F
# reparNum = 0  # Assuming reparNum is defined somewhere in your code
reparNum = pivoted_lab_df.rdd.getNumPartitions()
pivoted_lab_df_repart = pivoted_lab_df.repartition(reparNum)

▸,:,


In [42]:
from pyspark.sql.functions import col, count, when, lit

# Function to calculate the percentage of nulls in each column
def calculate_null_percentage(df):
    total_rows = df.count()
    null_counts = df.select([count(when(col(c).isNull(), 1)).alias(c) for c in df.columns])
    null_percentage = null_counts.select([(col(c) / total_rows).alias(c) for c in null_counts.columns])
    return null_percentage

# Calculate null percentages for each column
null_percentages_df = calculate_null_percentage(pivoted_lab_df_repart)

# Create a DataFrame to hold the threshold value
threshold_df = spark.createDataFrame([(0.9,)], ["threshold"])

# Create a condition to check which columns have null percentage greater than 90%
conditions = [when(col(c) > threshold_df.threshold, lit(c)).alias(c) for c in null_percentages_df.columns]

# Apply the conditions and filter out nulls to get the columns to drop
columns_to_drop_df = null_percentages_df.crossJoin(threshold_df).select(conditions)

# Collect the column names to drop from the DataFrame
columns_to_drop = [row[c] for row in columns_to_drop_df.collect() for c in columns_to_drop_df.columns if row[c] is not None]

# Drop those columns from the DataFrame
filtered_Control_df = pivoted_lab_df_repart.drop(*columns_to_drop)

# Display the result
print(f"Total number of columns with more than 90% nulls: {len(columns_to_drop)}")
print(f"Columns dropped: {columns_to_drop}")

# Show the new DataFrame
filtered_Control_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

Total number of columns with more than 90% nulls: 3180
Columns dropped: ['1005-8', '1006-6', '1007-4', '10328-3', '10329-1', '10333-3', '10334-1', '10335-8', '10338-2', '10346-5', '10362-2', '10368-9', '10374-7', '10376-2', '10378-8', '10379-6', '10380-4', '10381-2', '10449-7', '1048003', '10501-5', '10524-7', '10535-3', '10568-4', '10580-9', '10622-9', '10676-5', '10834-0', '10835-7', '10839-9', '10886-0', '10895-1', '10900-9', '10912-4', '10976-9', '10998-3', '11004-9', '11006-4', '11011-4', '11013-0', '11024-7', '11034-6', '11046-0', '11050-2', '11051-0', '11054-4', '11060-1', '11071-8', '11083-3', '11090-8', '11125-2', '11134-4', '11145-0', '11156-7', '11183-1', '11211-0', '11218-5', '11235-9', '11246-6', '11253-2', '11256-5', '11258-1', '11259-9', '11266-4', '11271-4', '11272-2', '11273-0', '11274-8', '11277-1', '11279-7', '11483-5', '11546-9', '11555-0', '11556-8', '11557-6', '11558-4', '11559-2', '11561-8', '11565-9', '11566-7', '11572-5', '11573-3', '11579-0', '11580-8', '11597

<IPython.core.display.Javascript object>

+------------------------------------+-------+-------+-------+------+------+------+-------+-------+------+------+------+-------+------+------+------+-------+------+------+--------+------+------+------+------+------+------+------+-------+-------+------+-------+-------+-------+--------+------+------+------+------+------+------+-------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+-------+
|personid                            |10466-1|13945-1|14979-9|1742-6|1751-7|1759-0|17861-6|19123-9|1920-8|1975-2|2028-9|20454-5|2075-0|2085-9|2093-3|21000-5|2160-0|2345-7|2514-8  |2571-8|2823-3|2885-2|2951-2|3040-3|3094-0|3097-3|32623-1|33037-3|4544-3|48642-3|48643-1|53115-2|5767-9  |5778-6|5794-3|5799-2|5802-4|5803-2|5811-5|58413-6|5902-2|5905-5|6301-6|6690-2|6768-6|704-7 |706-2 |711-2 |713-8 |718-7 |731-0 |736-9 |742-7 |751-8 |770-8 |777-3 |785-6 |786-4 |787-2 |788-0 |789-8 |94500-6|
+-----------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
# Get the list of column names
column_names = pivoted_lab_df.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 3243


In [43]:
from pyspark.sql.functions import col

# Perform left join
joined_df = final_df.join(filtered_Control_df, 
                                    final_df.personid == filtered_Control_df.personid,
                                    "left")

# Selecting columns from both DataFrames
selected_columns = [final_df[col] for col in final_df.columns] + \
                   [filtered_Control_df[col] for col in filtered_Control_df.columns if col != "personid"]

# Selecting columns from the joined DataFrame
result_df1 = joined_df.select(selected_columns)

▸,:,


In [35]:
result_df1.printSchema()

▸,:,


root
 |-- stratification_key: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- latest_diagdate: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Z21: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- X04: long (nullable = true)
 |-- F25: lo

In [36]:
# Get the list of column names
column_names = result_df1.columns

# Count the number of columns
num_columns = len(column_names)

# Display the number of columns
print("Number of columns:", num_columns)

▸,:,


Number of columns: 5555


In [44]:
result_df1.write.mode('overwrite').parquet('file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Epilepsy_Control_StratifiedSample2')

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
result_df1.printSchema()

▸,:,


root
 |-- stratification_key: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- TBI_date: string (nullable = true)
 |-- latest_diagdate: string (nullable = true)
 |-- age_of_TBI_diagnosis: double (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- MedicalHistory: integer (nullable = true)
 |-- M19: long (nullable = true)
 |-- S68: long (nullable = true)
 |-- Z21: long (nullable = true)
 |-- Y30: long (nullable = true)
 |-- A23: long (nullable = true)
 |-- B05: long (nullable = true)
 |-- H82: long (nullable = true)
 |-- V89: long (nullable = true)
 |-- R16: long (nullable = true)
 |-- I31: long (nullable = true)
 |-- Q61: long (nullable = true)
 |-- V72: long (nullable = true)
 |-- O12: long (nullable = true)
 |-- X76: long (nullable = true)
 |-- S39: long (nullable = true)
 |-- Z12: long (nullable = true)
 |-- L65: long (nullable = true)
 |-- X04: long (nullable = true)
 |-- F25: lo